# Detection of Plastic, Oil Spills and Micro Algae using YOLOv9
Batch 24 | CSE (Data Science) | Santhiram Engineering College

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.
Run the cells in order. Training checkpoints are saved to Google Drive so a disconnect will not lose your work.

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())

## 2. Install Ultralytics (supports YOLOv9, YOLOv8, YOLO11)

In [ ]:
!pip install -q ultralytics roboflow
import ultralytics
ultralytics.checks()

## 3. Mount Google Drive (to save results)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/water_pollution_yolo'
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Results will be saved in:", PROJECT_DIR)

## 4. Get the dataset
Pick **ONE** option.

**Option A - Roboflow Universe (easiest).** Find datasets for `oil spill`, `algae`, `floating plastic` on universe.roboflow.com, click *Download Dataset* -> format **YOLOv8** -> *show download code*, and paste the snippet below. A free Roboflow account is enough.

**Option B - Your own zip.** Upload a zip in YOLO format (folders `train/valid/test`, each with `images/` and `labels/`, plus a `data.yaml`) to Drive, and set the path.

In [ ]:
# ---------- OPTION A: Roboflow ----------
from roboflow import Roboflow
rf = Roboflow(api_key="YOUR_API_KEY")                 # <- paste your free key
project = rf.workspace("WORKSPACE_NAME").project("PROJECT_NAME")   # <- from the download snippet
dataset = project.version(1).download("yolov8")       # YOLOv8 format works for YOLOv9
DATA_YAML = f"{dataset.location}/data.yaml"
print(DATA_YAML)

In [ ]:
# ---------- OPTION B: your own zip on Drive ----------
# ZIP_PATH = '/content/drive/MyDrive/water_dataset.zip'
# !unzip -q "$ZIP_PATH" -d /content/dataset
# DATA_YAML = '/content/dataset/data.yaml'

### Merging several datasets
If oil, algae and plastic come from different datasets, merge them and make sure class IDs are consistent:
`0 = plastic`, `1 = oil_spill`, `2 = micro_algae`. Roboflow lets you remap/merge classes when you create a new project and import several sources.

## 5. Check / fix `data.yaml`

In [ ]:
import yaml
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)
print(cfg)
# If paths or class names look wrong, rewrite the file:
# cfg['names'] = ['plastic', 'oil_spill', 'micro_algae']
# cfg['nc'] = 3
# cfg['train'] = '/content/dataset/train/images'
# cfg['val']   = '/content/dataset/valid/images'
# cfg['test']  = '/content/dataset/test/images'
# with open(DATA_YAML, 'w') as f: yaml.safe_dump(cfg, f)

## 6. Quick look at the data

In [ ]:
import glob, random, cv2
import matplotlib.pyplot as plt

train_dir = os.path.join(os.path.dirname(DATA_YAML), 'train', 'images')
imgs = glob.glob(train_dir + '/*')
print("Training images:", len(imgs))

plt.figure(figsize=(14, 7))
for i, p in enumerate(random.sample(imgs, min(6, len(imgs)))):
    plt.subplot(2, 3, i+1)
    plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    plt.axis('off')
plt.show()

## 7. Smoke test (5 min)
Train a tiny model for a few epochs to confirm everything works before the long run.

In [ ]:
from ultralytics import YOLO
test_model = YOLO('yolov9t.pt')
test_model.train(data=DATA_YAML, epochs=3, imgsz=640, batch=16, project=PROJECT_DIR, name='smoke_test', exist_ok=True)

## 8. Full training - YOLOv9
Model sizes: `yolov9t` (tiny) < `yolov9s` < `yolov9m` < `yolov9c` < `yolov9e`.
Start with `yolov9s` or `yolov9c` on the free T4. If you get an out-of-memory error, lower `batch` to 8 or 4.

In [ ]:
model = YOLO('yolov9c.pt')       # change to yolov9s.pt if training is too slow

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # stop early if no improvement
    optimizer='auto',
    augment=True,
    mosaic=1.0,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,   # colour jitter helps with water/light changes
    fliplr=0.5,
    project=PROJECT_DIR,
    name='yolov9c_water',
    exist_ok=True,
    plots=True,
)

### If Colab disconnects, resume from the last checkpoint

In [ ]:
# model = YOLO(f'{PROJECT_DIR}/yolov9c_water/weights/last.pt')
# model.train(resume=True)

## 9. Evaluate on the test set

In [ ]:
best = YOLO(f'{PROJECT_DIR}/yolov9c_water/weights/best.pt')
metrics = best.val(data=DATA_YAML, split='test', imgsz=640)

print(f"Precision:     {metrics.box.mp:.3f}")
print(f"Recall:        {metrics.box.mr:.3f}")
print(f"mAP@0.5:       {metrics.box.map50:.3f}")
print(f"mAP@0.5:0.95:  {metrics.box.map:.3f}")
print("Per-class mAP@0.5:95:", dict(zip(best.names.values(), metrics.box.maps)))

## 10. Training curves, confusion matrix (use these in your report)

In [ ]:
from IPython.display import Image, display
run = f'{PROJECT_DIR}/yolov9c_water'
for f in ['results.png', 'confusion_matrix_normalized.png', 'PR_curve.png', 'F1_curve.png']:
    path = os.path.join(run, f)
    if os.path.exists(path):
        print(f); display(Image(path, width=800))

## 11. Predict on test images

In [ ]:
test_imgs = os.path.join(os.path.dirname(DATA_YAML), 'test', 'images')
preds = best.predict(source=test_imgs, conf=0.4, save=True, project=PROJECT_DIR, name='predictions', exist_ok=True)

pred_files = glob.glob(f'{PROJECT_DIR}/predictions/*.jpg')[:6]
plt.figure(figsize=(14, 8))
for i, p in enumerate(pred_files):
    plt.subplot(2, 3, i+1)
    plt.imshow(cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB))
    plt.axis('off')
plt.show()

## 12. Predict on your own image or video

In [ ]:
from google.colab import files
up = files.upload()                       # choose an image or video
src = list(up.keys())[0]
out = best.predict(source=src, conf=0.4, save=True, project=PROJECT_DIR, name='demo', exist_ok=True)
print("Saved in:", out[0].save_dir)

### Pollution alert logic (idea for your app)
Counts detections per class and prints an alert.

In [ ]:
def pollution_report(result, names):
    counts = {}
    for c in result.boxes.cls.tolist():
        counts[names[int(c)]] = counts.get(names[int(c)], 0) + 1
    for k, v in counts.items():
        print(f"ALERT: {k} detected ({v} instance(s))")
    if not counts:
        print("Water looks clear.")
    return counts

pollution_report(out[0], best.names)

## 13. Baseline comparison (YOLOv8 / YOLO11)
Same data, same settings. Put the results in one table in your report.

In [ ]:
baselines = {'yolov8s': 'yolov8s.pt', 'yolo11s': 'yolo11s.pt'}
rows = []
for name, w in baselines.items():
    m = YOLO(w)
    m.train(data=DATA_YAML, epochs=100, imgsz=640, batch=16, patience=20,
            project=PROJECT_DIR, name=name, exist_ok=True)
    r = YOLO(f'{PROJECT_DIR}/{name}/weights/best.pt').val(data=DATA_YAML, split='test')
    rows.append((name, r.box.mp, r.box.mr, r.box.map50, r.box.map))

rows.append(('yolov9c', metrics.box.mp, metrics.box.mr, metrics.box.map50, metrics.box.map))

import pandas as pd
df = pd.DataFrame(rows, columns=['Model', 'Precision', 'Recall', 'mAP@0.5', 'mAP@0.5:0.95']).round(3)
df

## 14. (Recommended) Segmentation for oil spills and algae
Oil and algae have irregular shapes, so polygon masks fit better than boxes. This needs a dataset labelled with **polygons** (Roboflow *Smart Polygon* tool helps). Then:

In [ ]:
# seg = YOLO('yolov9c-seg.pt')      # or 'yolov8s-seg.pt' / 'yolo11s-seg.pt'
# seg.train(data='path/to/seg_data.yaml', epochs=100, imgsz=640, batch=8,
#           project=PROJECT_DIR, name='seg_water', exist_ok=True)

## 15. Export for deployment (optional)

In [ ]:
best.export(format='onnx')        # for CPU / Raspberry Pi / web apps
# best.export(format='engine')    # TensorRT for Jetson

## 16. Download the trained weights

In [ ]:
from google.colab import files
files.download(f'{PROJECT_DIR}/yolov9c_water/weights/best.pt')